In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from prophet import Prophet

In [4]:
# Load dataset

df = pd.read_csv(r'C:\Users\ayoku\Downloads\dataset for workshop.csv.xls')

# Display first rows
print(df.head())

         date  temperature_c  humidity_pct  rainfall_mm  disease_cases
0  01/01/2023           22.5            68          2.1             34
1  02/01/2023           22.8            70          0.0             38
2  03/01/2023           23.1            72          5.3             45
3  04/01/2023           22.7            74          8.2             52
4  05/01/2023           23.4            69          1.0             41


In [5]:
# Convert date column to datetime

df['date'] = pd.to_datetime(df['date'], dayfirst=True)

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           365 non-null    datetime64[ns]
 1   temperature_c  365 non-null    float64       
 2   humidity_pct   365 non-null    int64         
 3   rainfall_mm    365 non-null    float64       
 4   disease_cases  365 non-null    int64         
dtypes: datetime64[ns](1), float64(2), int64(2)
memory usage: 14.4 KB
None


In [ ]:
print(df.describe())

In [ ]:
#Temperature Trend Visualization
plt.figure(figsize=(14,6), dpi=300)
plt.plot(df['date'], df['temperature_c'])
plt.title('Daily Temperature Trend')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.grid(True)
plt.show()

In [ ]:
#Temperature Trend Visualization
plt.figure(figsize=(14,6), dpi=300)
plt.plot(df['date'], df['humidity_pct'])
plt.title('Daily Humidity Trend')
plt.xlabel('Date')
plt.ylabel('Humidity(pct)')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(14,6), dpi=300)
plt.plot(df['date'], df['rainfall_mm'])
plt.title('Daily Rainfall Trend')
plt.xlabel('Date')
plt.ylabel('Rainfall (mm)')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(14,6), dpi=300)
plt.plot(df['date'], df['disease_cases'])
plt.title('Disease Cases Over Time')
plt.xlabel('Date')
plt.ylabel('Disease Cases')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8,6), dpi=300)

sns.heatmap(df[['temperature_c', 'humidity_pct',
                'rainfall_mm', 'disease_cases']].corr(),
            annot=True,
            cmap='coolwarm')

# Save image before showing
plt.savefig(
    'correlation_heatmap.png',
    dpi=300,
    bbox_inches='tight'
)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Create 7-day moving average

df['moving_avg'] = df['disease_cases'].rolling(window=7).mean()

plt.figure(figsize=(14,6), dpi=300)

plt.plot(df['date'], df['disease_cases'], label='Actual Cases')
plt.plot(df['date'], df['moving_avg'], label='7-Day Moving Average')

plt.title('Disease Cases Moving Average Forecast')
plt.xlabel('Date')
plt.ylabel('Disease Cases')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Linear Regression Forecasting
from sklearn.linear_model import LinearRegression

# Create time index

df['time_index'] = np.arange(len(df))

X = df[['time_index']]
y = df['disease_cases']

model = LinearRegression()
model.fit(X, y)

# Predictions

df['predicted_cases'] = model.predict(X)

In [ ]:
#Visualization of Linear Forecast
plt.figure(figsize=(14,6), dpi=300)

plt.plot(df['date'], df['disease_cases'], label='Actual')
plt.plot(df['date'], df['predicted_cases'],
         label='Linear Regression Forecast')

plt.title('Linear Regression Forecast')
plt.xlabel('Date')
plt.ylabel('Disease Cases')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
#Prophet requires columns named ds and y

prophet_df = df[['date', 'disease_cases']]
prophet_df.columns = ['ds', 'y']

print(prophet_df.head())

In [ ]:
from prophet import Prophet

model = Prophet()
model.fit(prophet_df)

In [ ]:
future = model.make_future_dataframe(periods=30)

forecast = model.predict(future)

print(forecast[['ds', 'yhat']].tail())

In [ ]:
fig1 = model.plot(forecast)
plt.title('Facebook Prophet Forecast')
plt.show()

In [ ]:
fig2 = model.plot_components(forecast)
plt.show()

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

# Build ARIMA model

arima_model = ARIMA(df['disease_cases'], order=(1,1,1))

arima_result = arima_model.fit()

forecast_arima = arima_result.forecast(steps=30)

print(forecast_arima)

In [ ]:
plt.figure(figsize=(14,6), dpi=300)

plt.plot(df['disease_cases'], label='Historical Data')
plt.plot(range(len(df), len(df)+30),
         forecast_arima,
         label='ARIMA Forecast')

plt.title('ARIMA Forecast')
plt.xlabel('Time')
plt.ylabel('Disease Cases')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

result = seasonal_decompose(df['disease_cases'],
                            model='additive',
                            period=30)

result.plot()
# Save image before showing
plt.savefig(
    'disease_cases.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

In [ ]:
fig = px.line(df,
              x='date',
              y='disease_cases',
              title='Interactive Disease Forecast')

fig.show()

In [ ]:
plt.figure(figsize=(14,7), dpi=300)

plt.plot(df['date'], df['temperature_c'], label='Temperature')
plt.plot(df['date'], df['humidity_pct'], label='Humidity')
plt.plot(df['date'], df['rainfall_mm'], label='Rainfall')

plt.legend()
plt.title('Environmental Variables Over Time')
plt.grid(True)
plt.show()

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data=[go.Scatter3d(
    x=df['temperature_c'],
    y=df['humidity_pct'],
    z=df['disease_cases'],
    mode='markers',
    marker=dict(
        size=5,
        color=df['rainfall_mm'],
        colorscale='Viridis'
    )
)])

fig.update_layout(
    title='3D Visualization of Environmental Factors and Disease Cases',
    scene=dict(
        xaxis_title='Temperature',
        yaxis_title='Humidity',
        zaxis_title='Disease Cases'
    )
)

fig.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np

z_data = np.random.rand(30,30)

fig = go.Figure(data=[go.Surface(z=z_data)])

fig.update_layout(
    title='3D Surface Forecast Visualization'
)

fig.show()

In [ ]:
from sklearn.metrics import mean_squared_error

rmse = np.sqrt(mean_squared_error(df['disease_cases'],
                                  df['predicted_cases']))

print('RMSE:', rmse)